# Day 3 — Monitoring and performance (Jupyter + AWS Console)

**Prerequisite:** Day 1 setup is complete — `start-lab.bat` has been run at least once, and `%CLUSTER_NAME%` is set.

On Day 2 you diagnosed an incident from **application logs** and `--describe`. Those show you what the *client* experienced. Today you add the **broker side of the story** from AWS CloudWatch, so you can answer the question every escalation eventually asks:

> While the application was failing, was the broker busy, out of disk, or completely idle?

### Two places you will work today

| Where | What you do there |
|-------|-------------------|
| **This notebook (CMD cells)** | Kafka CLI — describe the topic, generate load, measure consumer lag |
| **AWS Console (browser)** | CloudWatch **metrics**, CloudWatch **Logs**, **alarms**, and MSK cluster state |

### How today is sequenced, and why

1. **Confirm monitoring and logging are on** — a cluster only publishes what it has been configured to publish.
2. **Generate real traffic yourself** — CloudWatch can only draw what actually happened.
3. **Then read the graphs** — they will contain your own load, which means you can trust what you are looking at.

That order is deliberate. Opening a graph on an idle cluster shows a flat line, and a flat line teaches you nothing about troubleshooting.

Theory: [notes.md](notes.md). Command reference: [commands.md](commands.md).


## Setup — load the lab session

Every notebook cell starts a brand-new CMD process, so each cell begins by loading your lab variables:

`call ..\scripts\jupyter-lab-session.bat`

That sets `%CLUSTER_NAME%`, `%REGION%`, `%BOOTSTRAP%`, `%TOPIC%`, `%GROUP%` and `%CLIENT%` for the commands in that cell.

You need two of these values in the browser later, so note them from the output below:

- **`CLUSTER_NAME`** — you select this as the **Cluster Name** dimension on every CloudWatch metric
- **`REGION`** — the Console must be set to this region, or you will see no MSK metrics at all


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo CLUSTER_NAME=%CLUSTER_NAME%
echo REGION=%REGION%
echo TOPIC=%TOPIC%
echo GROUP=%GROUP%


## 0 — Confirm monitoring and logging are on

Before you can monitor anything, the cluster has to be **publishing** the data. MSK has **two separate switches**, and they are very often confused with each other:

| Switch | What it produces | Where it lands |
|--------|------------------|----------------|
| **Enhanced monitoring** | Numeric **metrics** — CPU, disk, bytes in and out, per broker | CloudWatch **Metrics**, namespace `AWS/Kafka` |
| **Broker log delivery** | The broker's own **log lines** | CloudWatch **Logs**, in a log group you choose |

Turning one on does **not** turn on the other. A cluster can have perfect metrics and no logs, or the other way round.

### Monitoring levels

Enhanced monitoring has levels, and the level decides which metrics exist at all:

| Level | What you get |
|-------|--------------|
| `DEFAULT` | Cluster-wide basics only. **No per-broker CPU or disk.** |
| `PER_BROKER` | Adds per-broker metrics: `CpuIdle`, `KafkaDataLogsDiskUsed`, `MemoryUsed` |
| `PER_TOPIC_PER_BROKER` | Adds a per-topic breakdown as well |
| `PER_TOPIC_PER_PARTITION` | Adds partition-level detail — the most detail and the highest cost |

This lab needs at least **`PER_BROKER`**, because nearly every real troubleshooting question is a per-broker question: *which broker is hot?*

**This is the honest reason a metric can come back with no data.** It is almost never a broken query. The cluster is simply not publishing that metric at the level it is configured for. Now that you know this, you can check it in ten seconds instead of doubting yourself.


### AWS Console — confirm the monitoring level

Both switches are already on for this cluster. Your task is to find them, read them, and record what they say — because on a real cluster this is the first thing you check when a metric looks missing, and you want to know exactly where it lives.

1. Open the **AWS Console** and set the region to match your `%REGION%` (top-right corner).
2. Go to **Amazon MSK** → **Clusters** → select **`%CLUSTER_NAME%`**.
3. Open the **Properties** tab and find the **Monitoring** panel.
4. Read the current **monitoring level** and write it down.

Record what you found:

| Setting | Value you observed |
|---------|--------------------|
| Monitoring level | |
| Broker log delivery enabled? | |
| Log group name | |

### How it would be changed

You will not change it here, but you should know where the control is, because raising the level is a routine request when a team says "we cannot see per-broker CPU".

The control is **Edit monitoring** on that same Properties tab. Changing the level does not restart the brokers, and new metrics begin appearing within a few minutes. It is not free — each level publishes more metrics, and CloudWatch charges per metric — which is why plenty of clusters sit at `DEFAULT` until somebody needs more.

**The takeaway for troubleshooting:** if `CpuIdle` is missing, check this panel before you doubt your query. A cluster at `DEFAULT` simply does not publish it.


### AWS Console — confirm broker log delivery

Broker logs answer a different question from metrics. Metrics tell you *how busy* the broker was. Logs tell you *what the broker was actually doing*.

1. In **Amazon MSK** → **Clusters** → **`%CLUSTER_NAME%`**, stay on the **Properties** tab.
2. Find the **Log delivery** panel.
3. Confirm **Deliver to Amazon CloudWatch Logs** is enabled, and note the **log group** name — it will look like `/aws/msk/%CLUSTER_NAME%`.
4. Add that log group name to the table above. You need it in section 1.

Once delivery is on, every broker writes its `server.log` content into that log group, with **one log stream per broker**. The control for turning it on or off is **Edit** on the same panel, and delivery begins within a few minutes of being enabled.

### Why both, on a real incident

A disk alarm fires. The metric tells you the broker is at 95% on its log volume. That is the *symptom*.

The broker log is what tells you whether it is deleting old segments normally, failing to delete them because retention is too long, or already rejecting writes. That is the *cause*. An escalation carrying both is resolved far faster than one carrying only a percentage.


## 0.2 — Generate real traffic so the graphs have data

CloudWatch draws what happened. If nothing happened, there is nothing to draw.

So before you open a single graph, you will put your own load on the cluster using **`kafka-producer-perf-test`** — a load generator that ships with Kafka and is the standard tool for exactly this job.

The next cell writes **2,000 messages of 1 KB each** (about 2 MB) to `%TOPIC%`, at a controlled rate of 1,000 messages per second. It takes a few seconds.

### What that load will move

| What the load creates | Metric it affects |
|-----------------------|-------------------|
| Messages written to the topic | `BytesInPerSec` rises |
| Work for the broker to do | `CpuIdle` dips |
| More data stored on the log volume | `KafkaDataLogsDiskUsed` rises slightly |
| Messages written but **not yet read** | `SumOffsetLag` rises — real, measurable consumer lag |

Notice the last row. You are deliberately **not** consuming yet. That creates genuine consumer lag on `%GROUP%`, which you will measure and then clear in section 2. Lag you created yourself is much easier to understand than lag you only read about.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-producer-perf-test.bat --topic %TOPIC% --num-records 2000 --record-size 1024 --throughput 1000 --producer.config %CLIENT% --producer-props bootstrap.servers=%BOOTSTRAP%


### Reading — the load generator output

The last line of the output is the summary that matters:

```
2000 records sent, 998.5 records/sec (0.98 MB/sec), 12.34 ms avg latency, 245.00 ms max latency
```

| Figure | What it tells you |
|--------|-------------------|
| `records/sec` and `MB/sec` | The throughput you actually achieved. Compare it with what the application team says they need. |
| `avg latency` | The typical time from send to acknowledgement — usually single-digit or low double-digit milliseconds on a healthy cluster. |
| `max latency` | The worst single send. A large gap between average and maximum means something paused: a leader change, a garbage-collection pause, or a slow disk. |

**Why this tool matters beyond the lab.** This is how you settle the most common argument in production: *"is Kafka slow, or is the application slow?"* Run the perf test from the same host as the application. If the perf test is fast and the application is still slow, the cluster is fine and the problem is in the application code or its configuration.

Your traffic is now on the cluster. MSK publishes metrics at **1-minute** granularity and they can take **2 to 3 minutes** to become visible, so carry straight on to the next section — the data will be waiting by the time you open the graphs.


### Find your partition leaders

Per-broker metrics are selected by **Broker ID**, so you need to know which brokers are actually holding your data.

Run the cell below and look at the **Leader** column. You will see three broker ids — for example `2`, `1`, `3`. Those are the brokers doing the work for your topic.

When you open a per-broker metric in the Console, start with the **leader of partition 0**, then add the other two so you can compare them.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


## 1 — Monitor Kafka metrics in CloudWatch

### Concept: `CpuIdle`

MSK reports **`CpuIdle`** — the percentage of CPU time the broker spent doing **nothing**. It is the *inverse* of how busy the broker is, which catches almost everyone out the first time.

| CpuIdle (Average) | How to read it |
|-------------------|----------------|
| **High** — around 70% or more | Plenty of headroom. CPU is not your bottleneck. |
| **Middle** — roughly 40 to 70% | Working, and healthy. |
| **Low** — under 20% | The broker is CPU-bound. Producers and consumers will start seeing latency. |

### Always compare all three brokers

One hot broker while the other two sit idle is a **skew** problem, not a capacity problem. It usually means too many partition leaders landed on one broker, or one partition is far busier than the rest.

The distinction changes your recommendation completely:

- **Skew** — rebalance partition leadership. Adding brokers will not help much.
- **Cluster-wide CPU pressure** — every broker is low on idle CPU. Now you are genuinely short of capacity.


### AWS Console — view CpuIdle

1. Open **CloudWatch**, with the region set to your `%REGION%`.
2. Go to **Metrics** → **All metrics**.
3. Choose the **`AWS/Kafka`** namespace.
4. Choose the dimension group **Broker ID, Cluster Name**.
5. Tick **`CpuIdle`** for the broker that leads partition 0. Tick the other two brokers as well so all three lines are on one graph.
6. Set **Statistic** to `Average` and **Period** to `1 minute`.
7. Set the time range to **Last 1 hour** using the range buttons at the top right.

**Use a relative range like "Last 1 hour", not a fixed date.** It always covers the load you just generated, whatever today's date happens to be, and it is what you would do on a live incident anyway.

Record what you see:

| Broker ID | CpuIdle (approx) | Busiest of the three? |
|-----------|------------------|------------------------|
| | | |
| | | |
| | | |


### Reading — what your CpuIdle graph is telling you

Look for two things.

**1. A dip that lines up with your load test.** You generated 2 MB of traffic a few minutes ago, so at least one broker should show a small dip in CpuIdle at that moment. That dip is *your own traffic*, and finding it proves the whole measurement chain works: your client → the broker → CloudWatch → this graph.

**2. The gap between the three brokers.** If one line sits consistently lower than the other two across the whole hour, that broker is carrying more work than its peers.

**If the line is flat and high, close to 100%:** that is a correct reading, not a failure. Your 2 MB burst is small next to a broker's capacity, so it barely registers. The *shape* of the line matters far more than the absolute number.

**If no data appears at all,** check these four things in order:

1. Is the Console in the same **region** as your `%REGION%`?
2. Did you pick the **Broker ID, Cluster Name** dimension group, rather than the cluster-level one?
3. Is the monitoring level at least **`PER_BROKER`**? `CpuIdle` does not exist below that level — see section 0.
4. Has enough time passed? Allow 2 to 3 minutes after generating load.


### AWS Console — read the broker logs

Now the other half of section 0: the broker's own log lines.

1. Open **CloudWatch** → **Logs** → **Log groups**.
2. Open the log group for your cluster, for example `/aws/msk/%CLUSTER_NAME%`.
3. You will see **one log stream per broker**. Open the stream for the broker you were just looking at.
4. Set the time range to **Last 1 hour** so it covers your load test.

### What broker logs contain

They are ordinary Kafka `server.log` lines. The ones worth recognising:

| Kind of line | What it means |
|--------------|---------------|
| Partition leadership changes | A leader moved from one broker to another. Clients briefly see `NOT_LEADER_OR_FOLLOWER` — exactly the error you read in the Day 2 producer log. |
| Segment rolled or deleted | Retention is doing its job, removing old data from disk. |
| ISR shrinking or expanding | A replica fell behind and later caught up. Sustained shrinking is a real problem. |
| Client rejected | An authentication or authorization failure — the broker side of Day 4's ACL errors. |

Record one line you found:

| Time | Level | What the broker said |
|------|-------|----------------------|
| | | |

**Metrics versus logs, in one sentence:** the metric tells you the broker was at 95% disk; the log tells you what it was doing about it.


### Concept: Alarms

A metric only helps if somebody is looking at it. Nobody watches graphs at 3 a.m. An **alarm** watches a metric for you and changes state when it crosses a line you defined, so a human gets told.

An alarm is a **notification, not a diagnosis.** It tells you that something crossed a threshold. It never tells you why. You still read the logs and still run `--describe`.

Every CloudWatch alarm has four parts:

| Part | Meaning | Example |
|------|---------|---------|
| **Metric** | What is being watched | `KafkaDataLogsDiskUsed` |
| **Threshold** | The line it must cross | greater than 80 percent |
| **Period and datapoints** | How long it must stay across the line before firing | 2 consecutive 5-minute periods |
| **Action** | Who gets told | an SNS topic that emails the on-call engineer |

### Why "datapoints" matters more than people expect

An alarm that fires on a single spike will fire constantly, and a team that gets constant alerts stops reading them. That is alert fatigue, and it is how real outages get missed.

Requiring **two or three consecutive periods** filters out momentary blips while still catching a genuine trend. Getting this number right is the difference between an alarm people trust and an alarm people mute.


### AWS Console — review the existing alarms

Before creating anything, find out what is already being watched. Duplicating an existing alarm is a common and unhelpful mistake.

1. Open **CloudWatch** → **Alarms** → **All alarms**.
2. Search for **`Kafka`**, or filter by the **`AWS/Kafka`** namespace if the Console offers it.
3. Open any alarm that mentions **`%CLUSTER_NAME%`**, disk, CPU, or lag.

Fill in one row per alarm you find:

| Alarm name | Metric | Threshold | State (OK / ALARM / INSUFFICIENT_DATA) |
|------------|--------|-----------|-----------------------------------------|
| | | | |

**A note on `INSUFFICIENT_DATA`.** This state means the alarm has not received enough datapoints to decide. It is not a failure, and it is not "OK" either — it means the alarm is currently blind. If you see it on a metric that should be flowing, that points straight back to the monitoring level in section 0.


### AWS Console — create alarms for the risks you found

Based on what you saw in **metrics** and **logs**, create **one alarm per critical risk**.

| # | Metric | The risk it catches | Example threshold |
|---|--------|---------------------|-------------------|
| 1 | **`CpuIdle`** | Broker running out of CPU headroom | Average **below 20** for **2** consecutive 5-minute periods |
| 2 | **`KafkaDataLogsDiskUsed`** | Log volume filling up | Average **above 80** percent |
| 3 | **`SumOffsetLag`** | A consumer group falling behind | Maximum **above 10000** — tune this to your topic's normal volume |

**Steps, repeated for each alarm:**

1. **CloudWatch** → **Alarms** → **Create alarm**.
2. **Select metric** → namespace **`AWS/Kafka`** → pick the metric from the table above.
3. Set the **dimensions**: Cluster Name always; add Broker ID for CPU and disk; add Consumer Group and Topic for lag.
4. **Conditions:** use the threshold from the table.
5. **Notification:** select an existing **SNS topic** if one exists for the class, otherwise choose **"No notification"** — for this lab the alarm state on its own is enough.
6. **Name:** make it descriptive enough to act on without opening it, for example `msk-class-cpu-low-broker2` or `msk-class-disk-high`.
7. **Create alarm**.

**Why these three metrics.** Between them they cover the three ways an MSK cluster actually hurts you: it runs out of **CPU**, it runs out of **disk**, or its **consumers stop keeping up**. Disk is the most urgent of the three — a full log volume stops the broker accepting writes altogether, and recovering from that takes far longer than preventing it.

Afterwards, confirm each alarm appears in the **Alarms** list and note the names for your assignment. If an alarm for that metric already exists on the cluster, do not create a second one — record the existing one instead.


## 2 — Analyze consumer lag

### Concept: what lag actually measures

Lag is a simple subtraction, but it is worth being precise about the three numbers:

| Column in `--describe` | Meaning |
|------------------------|---------|
| **`LOG-END-OFFSET`** | The offset of the next message the *producer* will write. How far the topic has got. |
| **`CURRENT-OFFSET`** | The last offset the *consumer group* has committed. How far the reader has got. |
| **`LAG`** | `LOG-END-OFFSET` minus `CURRENT-OFFSET` — how many messages are written but not yet processed. |

**Lag is measured per partition, per group.** A group can be perfectly healthy on two partitions and badly behind on a third. That single detail is what makes the difference between "the consumer is slow" and "one partition is hot", so always read the per-partition rows, never just the total.

### Slow consumer or slow producer?

The same complaint — "data is late" — has several different causes. These are the patterns that separate them:

| What you observe | Likely layer | Supporting evidence |
|------------------|--------------|---------------------|
| LAG growing, consumer log shows poll timeout | **Slow consumer** | `SumOffsetLag` rising while `BytesOutPerSec` stays flat |
| Producer `TimeoutException`, LAG flat | **Producer or network path** | `BytesInPerSec` flat; Day 2 `NOT_LEADER_OR_FOLLOWER` lines |
| `CpuIdle` low and disk high | **Broker capacity** | CpuIdle falling, `KafkaDataLogsDiskUsed` rising |
| Both BytesIn and BytesOut high, CPU fine | **Healthy load** | Compare against baseline; check for one hot partition |

### You already created lag on purpose

In section 0.2 you wrote 2,000 messages and deliberately did not read them. So `%GROUP%` is now genuinely behind, and the next cell will show you real numbers rather than a row of zeros.


### Measure the lag you created

The command below can take **30 to 90 seconds** over the public MSK endpoint, which is why it passes `--timeout 90000`. If it fails with `TimeoutException`, `LIST_OFFSETS`, or `FIND_COORDINATOR`, simply run the cell again — a first-attempt timeout on a public endpoint is common and is not a cluster fault.

**What you should see:** a row per partition, with `LAG` values that add up to roughly **2000** across the three partitions.

**If it reports that the group does not exist:** your consumer group has never been used on this machine. Run the drain cell two steps below once — that creates the group and commits a position — then come back and run this cell again.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


### Reading — the lag table

Write down the lag before you clear it. On a real ticket this is the number that quantifies the impact.

| Partition | CURRENT-OFFSET | LOG-END-OFFSET | LAG |
|-----------|----------------|----------------|-----|
| 0 | | | |
| 1 | | | |
| 2 | | | |
| **Total** | | | |

Two things to notice:

**The lag is spread across partitions, not identical on each.** The load generator distributes messages across all three partitions, so the split is roughly even but rarely exact. On a production topic with keyed messages the split can be very uneven, and that unevenness is itself a finding.

**`CONSUMER-ID` may show `-` or `none`.** That means no consumer is currently attached to the group. This is precisely the state that generates the ticket *"data stopped arriving"* — messages are accumulating and nothing is reading them. It is the same situation you recovered from in Day 2 section 4.


### Clear the lag and confirm

Now run a consumer to drain the backlog, then measure again.

The command sends the **message bodies** to `nul` — `> nul` discards standard output. Two thousand messages of random bytes would add megabytes of unreadable noise to this notebook, and you do not need to see them.

The useful line survives, because the consumer writes its summary to **standard error** rather than standard output. So you still get:

```
Processed a total of 2000 messages
```

**What you should see:** that summary line with a count close to 2,000, then `LAG` back to **0** on every partition.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\consume.bat --max-messages 2000 --timeout-ms 90000 > nul
echo === Lag after draining ===
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe --timeout 90000


### Reading — before versus after

| Column | Before draining | After draining |
|--------|-----------------|----------------|
| `LAG` | around 2000 in total | **0** on every partition |
| `CURRENT-OFFSET` | behind `LOG-END-OFFSET` | equal to `LOG-END-OFFSET` |
| `CONSUMER-ID` | `-` or `none` | a real consumer id, or `-` again once the cell finished |

You have now watched a full lag cycle: lag appeared because nothing was reading, and it went to zero because something read. That is the entire shape of the most common Kafka support ticket there is.

**If LAG is still above 0:** the consumer stopped at its message limit before finishing. Run the cell again — each run continues from the committed position, so nothing is lost and nothing is duplicated.

**The judgement call that matters.** Lag on its own is not an incident. Lag that keeps **growing** is. A backlog of 2,000 that drains in ten seconds is fine; a backlog of 2,000 that was 500 an hour ago is a consumer that cannot keep up with its input, and no amount of restarting will fix that — it needs more consumer instances, or faster processing per message.


### AWS Console — view the consumer lag metric

Kafka CLI gives you lag *right now*. CloudWatch gives you lag *over time*, which is what tells you whether it is growing.

1. **CloudWatch** → **Metrics** → **All metrics** → **`AWS/Kafka`**.
2. Find **`SumOffsetLag`**. If the Console does not list it, use **`MaxOffsetLag`** instead.
3. Set the dimensions:
   - **Cluster Name** = `%CLUSTER_NAME%`
   - **Consumer Group** = `%GROUP%`
   - **Topic** = `%TOPIC%`
4. Time range **Last 1 hour**, Statistic **Maximum**, Period **1 minute**.

**What the shape should look like:** a rise while you were producing without consuming, then a drop back to zero when you drained it. That hill is your incident, start to finish, in one picture.

Record the peak value:

| Peak `SumOffsetLag` | Time it peaked | Time it returned to zero |
|---------------------|----------------|---------------------------|
| | | |

**Why `Maximum` and not `Average`:** you care about the worst moment, not the typical one. Averaging a lag spike across an hour hides exactly the thing you are looking for.


## 3 — Performance bottlenecks (throughput)

### Concept: BytesIn and BytesOut

| Metric | What it measures |
|--------|------------------|
| **`BytesInPerSec`** | Bytes arriving from producers — the write load |
| **`BytesOutPerSec`** | Bytes leaving to consumers — the read load |

### Reading them as a pair

Neither number means much alone. The **relationship** between them is what identifies the bottleneck:

| BytesIn | BytesOut | What it usually means |
|---------|----------|-----------------------|
| Rising | Rising, roughly in step | Healthy pipeline. Consumers are keeping up with producers. |
| Rising | Flat or falling | **Consumers are not keeping up.** Expect lag to be growing at the same moment. |
| Flat | Flat, while the app reports errors | The traffic never reached the broker. Suspect the **network path**, the bootstrap address, or authentication — not cluster capacity. |
| Falling sharply | Falling sharply | Producers stopped. Look upstream at the application, not at Kafka. |

That third row is the one that saves you the most time. In Day 2 the producer logged `TimeoutException` after `TimeoutException`, and a flat `BytesInPerSec` during that window would prove the messages never arrived at all. There is no point tuning a broker that never received the traffic.

**Expect `BytesOutPerSec` to be higher than you first guess.** Every message is read once per consumer group, and replication traffic between brokers counts too, so a topic with three consumer groups sends its data out several times over.


### AWS Console — throughput metrics

1. **CloudWatch** → **Metrics** → **All metrics** → **`AWS/Kafka`**.
2. Start at **cluster level** — the dimension group with **Cluster Name** only.
3. Add both **`BytesInPerSec`** and **`BytesOutPerSec`** to the same graph so you can compare their shapes directly.
4. Time range **Last 1 hour**, Statistic **Average**, Period **1 minute**.

**What you should see:** a clear spike in `BytesInPerSec` when you ran the load generator, followed by a spike in `BytesOutPerSec` when you drained the backlog. Two humps, slightly offset in time — write first, read second.

That offset *is* consumer lag, drawn as a picture. The horizontal gap between the two humps is how long the data sat unread.

Record what you measured:

| Metric | Peak value | Time of peak |
|--------|-----------|--------------|
| `BytesInPerSec` | | |
| `BytesOutPerSec` | | |

To see which broker took the load, add the **Broker ID** dimension and put all three brokers on the graph. An even split across three brokers is what you want; one broker taking most of the traffic points to partition skew.


### Reading — latency versus timeout

The single log line **`TimeoutException`** covers two completely different failures, and telling them apart is the whole job:

| Which failure | What actually happened | Where to look first |
|---------------|------------------------|---------------------|
| **Never connected** | Wrong bootstrap address, wrong port, or a Security Group blocking the path. No bytes ever reached the broker. | Fix the client's network path. `BytesInPerSec` stays flat throughout. |
| **Connected but too slow** | The broker is under load, or `acks=all` is waiting on a replica that has fallen out of sync. Bytes are flowing, just not fast enough. | `CpuIdle`, disk, and the ISR column in `--describe`. |

The distinguishing evidence is `BytesInPerSec`. Flat means never connected. Non-zero but struggling means connected and slow. Those two findings lead to completely different teams and completely different fixes, which is why it is worth thirty seconds to check.

**One practical note about ports in this lab:** your machine connects over the public endpoint on port **9196**. An application running inside the VPC would use **9096** instead. When someone reports a timeout, confirm which path their client is actually using before assuming anything about the cluster.


## 4 — Broker disk and cluster state

### Concept: `KafkaDataLogsDiskUsed`

This is the percentage used of the **Kafka log volume** — the EBS volume where MSK stores topic data. It is **not** your Windows `C:` drive, and it is not the broker's root volume.

| Disk used | What it means for you |
|-----------|----------------------|
| Under 60% | Comfortable |
| 60 to 80% | Watch the trend. Is it climbing, and how fast? |
| Above 80% | Act now. This is the alarm you set in section 1. |
| At 100% | The broker **stops accepting writes**. Producers fail, replication stalls, and recovery is slow and manual. |

Disk is the most unforgiving MSK metric. High CPU makes things slow; a full log volume makes them stop. That asymmetry is why disk deserves the tightest alarm of the three.

### Concept: what "retention" means

**Retention** is how long Kafka keeps a message before deleting it — whether or not anyone has read it. Two topic settings control it:

| Setting | Meaning |
|---------|---------|
| `retention.ms` | Delete messages older than this age |
| `retention.bytes` | Once a partition grows past this size, delete its oldest data |

Whichever limit is reached first wins. Kafka deletes whole **segment files** rather than individual messages, so disk usage falls in steps rather than smoothly.

**Why retention comes up in a disk conversation.** Disk usage and retention are the same problem seen from opposite ends. If `KafkaDataLogsDiskUsed` keeps climbing, either more data is arriving than before, or retention is keeping data for longer than the volume can hold. You have two levers: store less, or keep it for less time.

**Why retention comes up in a recovery conversation.** You can only re-read messages that are **still on disk**. Once retention has deleted them, no offset reset and no replay will bring them back. This is the single most important constraint on message recovery, and it is why the first question on any "can we replay yesterday's data?" ticket is *"what is the retention on that topic?"*

**Today you only observe.** Actually changing `retention.ms` on a topic with `kafka-configs.bat` is a Day 4 exercise — that is what "retention changes are Day 4" means. Today you read the disk graph and form an opinion about whether retention is appropriate for the volume.


### AWS Console — broker disk and cluster state

**Disk usage per broker:**

1. **CloudWatch** → **Metrics** → **All metrics** → **`AWS/Kafka`**.
2. Choose the **Broker ID, Cluster Name** dimension group.
3. Select **`KafkaDataLogsDiskUsed`** for all three brokers.
4. Time range **Last 1 hour**, Statistic **Average**, Period **1 minute**.

**What you should see:** three lines at a similar level, with a very small step upward where your 2 MB of load landed. Two MB is tiny next to an EBS volume, so expect a nudge rather than a jump.

**What to look for beyond the number:** the three brokers should sit close together. One broker noticeably higher than the others means data is not spread evenly — usually partition skew, sometimes a partition whose replicas landed badly.

| Broker ID | Disk used % | Notably different from the others? |
|-----------|-------------|------------------------------------|
| | | |
| | | |
| | | |

**Cluster state:**

1. **Amazon MSK** → **Clusters** → **`%CLUSTER_NAME%`**.
2. Confirm **State** is **Active**.

The states you might see, and what they mean:

| State | Meaning |
|-------|---------|
| **Active** | Normal operation |
| **Updating** | A configuration change or version upgrade is in progress. Brokers restart one at a time, so clients may see brief leader changes. |
| **Healing** | AWS is replacing a failed broker. Expect under-replicated partitions until it finishes. |
| **Maintenance** | AWS-initiated patching |

**Healing is worth recognising on sight.** It explains under-replicated partitions and brief produce failures without anyone having done anything wrong — and it means the correct action is to wait and monitor, not to start changing configuration.


### Reading — putting the four metrics together

You now have four independent views of the same cluster. The skill being practised is reading them **together**, because any one of them alone will mislead you.

| CpuIdle | Disk | BytesIn | Lag | The story these four tell |
|---------|------|---------|-----|---------------------------|
| High | Low | Flat | Flat | Idle and healthy. If the app is failing, the problem is not the broker. |
| High | Low | Flat | **Growing** | Producers stopped, or nothing is consuming. Look at the applications. |
| Low | Low | High | Low | Genuinely busy and coping. Consider capacity if this is the new normal. |
| High | **High** | Flat | Flat | Retention is holding too much data for the volume. Disk is the risk, not load. |
| **Low** | High | High | **Growing** | The broker is saturated. This is a real capacity incident. |

**The habit to take away:** never escalate on a single metric. "CPU is high" invites an argument. "CpuIdle dropped to 15% on broker 2 only, disk is steady at 45%, BytesIn doubled at 14:05, and lag on `%GROUP%` has grown from 0 to 40,000 since then" invites a fix.


## RCA worksheet (required)

Write **4 to 6 sentences** using the Day 2 log evidence and the metrics you measured today.

| # | What to write | Where your evidence came from |
|---|---------------|-------------------------------|
| 1 | **Symptom** — what failed, in the words a user would use | Day 2 application logs |
| 2 | **Time window** — first and last error timestamp, in UTC | Day 2 application logs |
| 3 | **Broker evidence** — CpuIdle, disk, BytesIn and BytesOut for that window | CloudWatch metrics, today |
| 4 | **Probable layer** — application, consumer, broker, or network path | The combination, not one metric |
| 5 | **One next step** — the single action you would take first | Your judgement |
| 6 | **What you would not do** — and why | Your judgement |

That last row is the one that marks out someone who has done this before. Knowing that you do **not** reset offsets to fix consumer lag, and being able to say why, is worth more than knowing twenty commands.

Paste your answer into [samples/assignment-metrics.md](samples/assignment-metrics.md).


### Lab complete? — quick checklist

| TOC item | Where you did it |
|----------|------------------|
| Monitor Kafka metrics in CloudWatch | Console — CpuIdle, disk, BytesIn and BytesOut, all on live data |
| Check broker logs | Console — CloudWatch Logs, one stream per broker |
| Set up alarms | Console — reviewed existing alarms, then created alarms for CPU, disk, and lag |
| Analyze consumer lag | Jupyter — created lag, measured it, drained it; Console — `SumOffsetLag` over time |
| Identify performance bottlenecks | `kafka-producer-perf-test` throughput and latency, plus the BytesIn/BytesOut pair |
| Verify broker resource utilization | Console — disk per broker, and MSK cluster State = Active |

**The two rules of thumb from today:**

- Fix **consumer lag** by getting the consumer working again — more instances, faster processing, or a restart. Resetting offsets is not a lag fix; it is a data decision, and it belongs to Day 4.
- Fix **disk** by reducing retention or adding capacity, and alarm on it before it reaches 100%. A full log volume stops writes completely.


## Assignment

Fill in [samples/assignment-metrics.md](samples/assignment-metrics.md) using the values you measured today.

| Source | What you take from it |
|--------|-----------------------|
| **Kafka CLI** in this notebook | Topic partition leaders and ISR; lag before and after draining; perf-test throughput and latency |
| **CloudWatch Metrics** | CpuIdle per broker, `KafkaDataLogsDiskUsed`, `BytesInPerSec` and `BytesOutPerSec`, peak `SumOffsetLag` |
| **CloudWatch Logs** | One broker log line, with its timestamp and level |
| **MSK console** | Monitoring level, log delivery setting, cluster State |
| **Alarms you created** | Alarm names and thresholds |

**Example of the standard expected:**

> On topic `orders-user15` the three partitions are led by brokers 2, 1 and 3, all with full ISR. A 2,000-message load test achieved 998 records/sec at 12 ms average latency, with a 245 ms maximum — the gap suggests a brief pause rather than sustained pressure. `BytesInPerSec` peaked at 33 KB/s at 09:42 UTC, and `BytesOutPerSec` peaked three minutes later when I drained the backlog. `SumOffsetLag` on `cg-user15-support` peaked at 2,000 and returned to 0 at 09:45. CpuIdle stayed above 90% on all three brokers and disk held steady at 41%, so the cluster had ample headroom throughout. I created `msk-class-disk-high` at 80% and `msk-class-cpu-low-broker2` below 20% for two periods.
